# NusaSR v2: fine-tune untuk kondisi lapangan

Lanjutan dari v1 (`NusaSR_FineTune_Colab.ipynb`). Kerusakan LoRa tetap sama persis dengan app: diperkecil ke 160/256/384 px lalu WebP 2/4/8 KB. Yang baru:

1. **Kondisi foto lapangan disimulasikan** sebelum dikirim: gelap + noise sensor, kabut/asap, goyang (motion blur), silau, warna melenceng, lalu disimpan JPEG seperti kamera HP. Kunci jawabannya foto yang sudah berkondisi itu juga, jadi model belajar memperjelas, bukan menghapus asap atau menerangkan malam.
2. **Foto HP sendiri ikut dilatih** (opsional): taruh di Drive `NusaSR/foto_latih`. Foto ini diambil ±30% dari setiap batch.
3. **Mulai dari model v1** (`NusaSR/best.pt`) kalau ada, jadi tidak mengulang dari nol.
4. **Uji lebih lengkap:** Kodak 24 foto di setiap kondisi, DIV2K valid, dan foto lapangan `NusaSR/foto_lapangan` (hanya uji, tidak dilatih). Ditambah ukuran kerugian kuantisasi 8-bit dan gambar contoh.

**Cara pakai:** Runtime → Change runtime type → **GPU**, lalu Run all. Hasil dan checkpoint di Drive folder `NusaSR2`, terpisah dari v1. Kalau Colab putus, Run all lagi dan training lanjut.

Foto di `foto_latih` dan `foto_lapangan` harus beda tempat/kejadian. Foto yang sama persis otomatis dibuang dari data latih. Format JPG/PNG/WEBP; HEIC dari iPhone belum didukung.

## 1. Persiapan

In [1]:
# COLAB_ONLY
!pip install -q lpips scikit-image
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# CONFIG
import glob, hashlib, io, math, os, random, time, urllib.request, zipfile
from multiprocessing.pool import ThreadPool
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageDraw, ImageOps
from torch.utils.data import DataLoader, Dataset

WORK = Path("/content/nusasr2")
SAVE = Path("/content/drive/MyDrive/NusaSR2")
V1_DIR = Path("/content/drive/MyDrive/NusaSR")
V1_CKPT = V1_DIR / "best.pt"
OWN_HR = V1_DIR / "foto_latih"        # foto HP sendiri untuk dilatih (opsional)
FIELD_HR = V1_DIR / "foto_lapangan"   # foto uji, tidak ikut dilatih (opsional)
TRAIN_HR = WORK / "DIV2K_train_HR"
VAL_HR = WORK / "DIV2K_valid_HR"
KODAK_HR = WORK / "kodak"
WEIGHTS = WORK / "realesr-general-x4v3.pth"

PRESETS = [(160, 2000), (256, 4000), (384, 8000)]  # (sisi terpanjang px, batas byte), sama dengan LoraImageQuality
CONDITIONS = ("gelap", "kabut", "goyang", "silau", "warna")
VARIANTS = 2            # per foto latih: 1 asli + 1 kondisi lapangan acak
OWN_SHARE = 0.3         # porsi batch dari foto_latih bila ada
SCALE = 4
LR_PATCH = 64
ITERS = 30000
BATCH = 16
LR_FROM_V1 = 5e-5       # learning rate bila mulai dari model v1
LR_FROM_BASE = 1e-4     # learning rate bila mulai dari bobot bawaan
NUM_WORKERS = 2
VAL_EVERY = 2000
VAL_PAIRS = 90
TFLITE_IMAGES = 10
CALIBRATION = 300
OVERLAP = 16            # sama dengan SrTiling.OVERLAP di app
THREADS = 4
SEED = 42
SAMPLES = ("kodim05", "kodim20", "kodim23")

In [3]:
for d in (WORK, SAVE):
    d.mkdir(parents=True, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(tanpa GPU: ganti runtime ke GPU)")

device: cuda Tesla T4


In [4]:
# COLAB_ONLY
def fetch(url, dest):
    """Unduh ke file .part dulu, jadi unduhan yang terputus tidak dianggap selesai."""
    dest = Path(dest)
    if dest.exists():
        return
    part = dest.with_name(dest.name + ".part")
    shown = [-1]

    def hook(blocks, block_size, total):
        if total > 50e6:
            step = min(100, blocks * block_size * 100 // total) // 10
            if step > shown[0]:
                shown[0] = step
                print(f"  {step * 10}% dari {total / 1e9:.2f} GB", flush=True)

    print("unduh", url, flush=True)
    urllib.request.urlretrieve(url, part, hook)
    part.rename(dest)


BASE = "http://data.vision.ee.ethz.ch/cvl/DIV2K"
for name in ("DIV2K_train_HR", "DIV2K_valid_HR"):
    if not (WORK / name).exists():
        z = WORK / f"{name}.zip"
        fetch(f"{BASE}/{name}.zip", z)
        with zipfile.ZipFile(z) as f:
            f.extractall(WORK)
        z.unlink()
fetch("https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth", WEIGHTS)
KODAK_HR.mkdir(parents=True, exist_ok=True)
for i in range(1, 25):
    fetch(f"https://r0k.us/graphics/kodak/kodak/kodim{i:02d}.png", KODAK_HR / f"kodim{i:02d}.png")
print("data siap")

unduh http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip
  0% dari 3.53 GB
  10% dari 3.53 GB
  20% dari 3.53 GB
  30% dari 3.53 GB
  40% dari 3.53 GB
  50% dari 3.53 GB
  60% dari 3.53 GB
  70% dari 3.53 GB
  80% dari 3.53 GB
  90% dari 3.53 GB
  100% dari 3.53 GB
unduh http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_valid_HR.zip
  0% dari 0.45 GB
  10% dari 0.45 GB
  20% dari 0.45 GB
  30% dari 0.45 GB
  40% dari 0.45 GB
  50% dari 0.45 GB
  60% dari 0.45 GB
  70% dari 0.45 GB
  80% dari 0.45 GB
  90% dari 0.45 GB
  100% dari 0.45 GB
unduh https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth
unduh https://r0k.us/graphics/kodak/kodak/kodim01.png
unduh https://r0k.us/graphics/kodak/kodak/kodim02.png
unduh https://r0k.us/graphics/kodak/kodak/kodim03.png
unduh https://r0k.us/graphics/kodak/kodak/kodim04.png
unduh https://r0k.us/graphics/kodak/kodak/kodim05.png
unduh https://r0k.us/graphics/kodak/kodak/kodim06.png
unduh https://r0k.us/graphics/k

## 2. Model

In [5]:
class SRVGGNetCompact(nn.Module):
    """Arsitektur Real-ESRGAN General x4v3 (32 konvolusi, 64 fitur, PReLU)."""
    def __init__(self, feats=64, convs=32, scale=SCALE):
        super().__init__()
        self.scale = scale
        body = [nn.Conv2d(3, feats, 3, 1, 1), nn.PReLU(feats)]
        for _ in range(convs):
            body += [nn.Conv2d(feats, feats, 3, 1, 1), nn.PReLU(feats)]
        body.append(nn.Conv2d(feats, 3 * scale * scale, 3, 1, 1))
        self.body = nn.ModuleList(body)
        self.upsampler = nn.PixelShuffle(scale)

    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        return self.upsampler(out) + F.interpolate(x, scale_factor=self.scale, mode="nearest")


def load_pretrained():
    model = SRVGGNetCompact()
    model.load_state_dict(torch.load(WEIGHTS, map_location="cpu", weights_only=True)["params"])
    return model


def load_v1():
    model = SRVGGNetCompact()
    model.load_state_dict(torch.load(V1_CKPT, map_location="cpu", weights_only=True))
    return model


HAS_V1 = V1_CKPT.exists()
print("model v1:", "ada" if HAS_V1 else "tidak ada, mulai dari bobot bawaan")

model v1: ada


## 3. Kondisi lapangan
Diterapkan ke foto resolusi penuh, sebelum diperkecil. Setiap kondisi memakai angka acak yang tetap per foto, jadi hasilnya sama setiap kali notebook dijalankan.

In [6]:
def seed_of(text):
    return int(hashlib.md5(text.encode()).hexdigest()[:8], 16)


def apply_condition(img, kind, seed):
    if kind == "asli":
        return img
    rng = np.random.default_rng(seed)
    a = np.asarray(img, dtype=np.float32) / 255.0
    if kind == "gelap":  # malam atau dalam ruangan tanpa lampu, dengan noise sensor
        a = a ** rng.uniform(1.2, 2.0) * rng.uniform(0.25, 0.6)
        read, shot = rng.uniform(0.004, 0.015), rng.uniform(0.01, 0.03)
        a = a + rng.standard_normal(a.shape, dtype=np.float32) * (read + shot * np.sqrt(a))
    elif kind == "kabut":  # asap, debu, hujan rapat
        t = rng.uniform(0.45, 0.8)
        air = rng.uniform(0.7, 0.95) + rng.uniform(-0.03, 0.03, 3)
        a = a * t + air * (1 - t)
    elif kind == "goyang":  # tangan goyang saat memotret
        length = max(3, int(max(a.shape[:2]) * rng.uniform(0.004, 0.012)))
        angle = rng.uniform(0, math.pi)
        acc = np.zeros_like(a)
        for i in range(length):
            d = i - (length - 1) / 2
            acc += np.roll(a, (int(round(d * math.sin(angle))), int(round(d * math.cos(angle)))), axis=(0, 1))
        a = acc / length
    elif kind == "silau":  # terlalu terang atau melawan cahaya
        a = 1 - (1 - a) ** rng.uniform(1.6, 2.6)
    elif kind == "warna":  # white balance kamera melenceng
        gain = rng.uniform(0.8, 1.2, 3)
        a = a * (gain / gain.mean())
    out = Image.fromarray((np.clip(a, 0, 1) * 255).round().astype(np.uint8))
    buf = io.BytesIO()
    out.save(buf, "JPEG", quality=int(rng.integers(75, 96)))  # disimpan seperti kamera HP
    return Image.open(io.BytesIO(buf.getvalue())).convert("RGB")


def random_kinds(stem):
    kinds = ["asli"]
    for v in range(1, VARIANTS):
        kinds.append(CONDITIONS[seed_of(f"{stem}-{v}") % len(CONDITIONS)])
    return kinds


def all_kinds(stem):
    return ["asli", *CONDITIONS]


def only_original(stem):
    return ["asli"]

## 4. Kerusakan seperti di app
Mengikuti `LoraImageCodec.kt`: `inSampleSize` pangkat 2, resize bilinear, lalu kualitas WebP tertinggi yang muat batas byte. Bila kualitas 5 pun tidak muat, sisi gambar diperkecil 15%.

In [7]:
QUALITY_MIN, QUALITY_MAX, MIN_SIDE, SIDE_SHRINK = 5, 90, 64, 0.85


def android_downscale(src, target):
    long_side = max(src.size)
    sample = 1
    while long_side // (sample * 2) >= target * 2:
        sample *= 2
    img = src.reduce(sample) if sample > 1 else src
    s = target / max(img.size)
    return img.resize((max(1, int(img.width * s + 0.5)), max(1, int(img.height * s + 0.5))), Image.BILINEAR)


def webp_bytes(img, quality):
    buf = io.BytesIO()
    img.save(buf, "WEBP", quality=quality)
    return buf.getvalue()


def lora_degrade(src, max_side, max_bytes):
    side = max_side
    while True:
        small = android_downscale(src, side)
        lo, hi, best = QUALITY_MIN, QUALITY_MAX, None
        while lo <= hi:
            mid = (lo + hi) // 2
            data = webp_bytes(small, mid)
            if len(data) <= max_bytes:
                best, lo = data, mid + 1
            else:
                hi = mid - 1
        if best is not None or side <= MIN_SIDE:
            data = best if best is not None else webp_bytes(small, QUALITY_MIN)
            return Image.open(io.BytesIO(data)).convert("RGB")
        side = max(MIN_SIDE, int(side * SIDE_SHRINK))


def make_pairs(args):
    """Satu foto + satu kondisi -> pasangan untuk ketiga preset."""
    path, out, kind, allow_small = args
    stem = Path(path).stem.replace("_", "-").replace(" ", "-")
    names = [(out / f"{stem}-{kind}_{side}_lr.npy", side, size) for side, size in PRESETS]
    if all(p.exists() and Path(gt_path(str(p))).exists() for p, _, _ in names):
        return [str(p) for p, _, _ in names]
    try:
        src = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    except Exception as e:
        print("lewati", path, e)
        return []
    src = apply_condition(src, kind, seed_of(f"{stem}-{kind}"))
    made = []
    for lr_path, side, size in names:
        lr = lora_degrade(src, side, size)
        want = (lr.width * SCALE, lr.height * SCALE)
        if want[0] > src.width or want[1] > src.height:
            if not allow_small:
                continue  # foto terlalu kecil untuk jadi kunci jawaban latih
            gt = src  # uji: hasil diperkecil ke ukuran foto asli saat dinilai
        else:
            gt = src.resize(want, Image.LANCZOS)
        if not allow_small and min(lr.size) < LR_PATCH:
            continue
        np.save(gt_path(str(lr_path)), np.asarray(gt, dtype=np.uint8))
        np.save(lr_path, np.asarray(lr, dtype=np.uint8))
        made.append(str(lr_path))
    return made


def image_files(folder):
    files = []
    for ext in ("png", "jpg", "jpeg", "webp"):
        for e in (ext, ext.upper()):
            files += glob.glob(f"{folder}/**/*.{e}", recursive=True)
    return sorted(set(files))


def build_pairs(files, out, kinds_of, allow_small=False):
    out.mkdir(parents=True, exist_ok=True)
    jobs = [(p, out, k, allow_small) for p in files for k in kinds_of(Path(p).stem.replace("_", "-"))]
    with ThreadPool(os.cpu_count() or 2) as pool:
        made = [x for group in pool.imap_unordered(make_pairs, jobs) for x in group]
    print(f"{out.name}: {len(files)} foto, {len(made)} pasangan", flush=True)
    return sorted(made)


def gt_path(lr_path):
    return lr_path[:-len("lr.npy")] + "gt.npy"


def preset_of(lr_path):
    return int(Path(lr_path).stem.rsplit("_", 2)[1])


def kind_of(lr_path):
    return Path(lr_path).stem.rsplit("_", 2)[0].rsplit("-", 1)[1]


def file_hash(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


field_files = image_files(FIELD_HR) if FIELD_HR.exists() else []
own_files = image_files(OWN_HR) if OWN_HR.exists() else []
field_hashes = {file_hash(p) for p in field_files}
leaked = [p for p in own_files if file_hash(p) in field_hashes]
if leaked:
    print(f"{len(leaked)} foto di foto_latih sama persis dengan foto_lapangan, dibuang dari data latih:", [Path(p).name for p in leaked])
own_files = [p for p in own_files if p not in leaked]

t0 = time.time()
train_pairs = build_pairs(image_files(TRAIN_HR), WORK / "pairs_train", random_kinds)
own_pairs = build_pairs(own_files, WORK / "pairs_own", random_kinds) if own_files else []
val_pairs = build_pairs(image_files(VAL_HR), WORK / "pairs_val", random_kinds)
kodak_pairs = build_pairs(image_files(KODAK_HR), WORK / "pairs_kodak", all_kinds, allow_small=True)
field_pairs = build_pairs(field_files, WORK / "pairs_field", only_original, allow_small=True) if field_files else []
assert train_pairs and val_pairs, "dataset kosong"
print(f"pasangan siap dalam {(time.time() - t0) / 60:.1f} menit")

pairs_train: 800 foto, 4790 pasangan
pairs_val: 100 foto, 600 pasangan
pairs_kodak: 24 foto, 432 pasangan
pasangan siap dalam 15.4 menit


In [8]:
def to_tensor(a):
    return torch.from_numpy(a.copy()).permute(2, 0, 1).float() / 255.0


class Patches(Dataset):
    def __init__(self, groups, length):
        self.groups = [(pairs, w) for pairs, w in groups if pairs]
        self.weights = [w for _, w in self.groups]
        self.length = length

    def __len__(self):
        return self.length

    def __getitem__(self, _):
        pairs = random.choices([g for g, _ in self.groups], self.weights)[0]
        p = random.choice(pairs)
        lr, gt = np.load(p, mmap_mode="r"), np.load(gt_path(p), mmap_mode="r")
        y, x = random.randint(0, lr.shape[0] - LR_PATCH), random.randint(0, lr.shape[1] - LR_PATCH)
        a = np.array(lr[y:y + LR_PATCH, x:x + LR_PATCH])
        b = np.array(gt[y * SCALE:(y + LR_PATCH) * SCALE, x * SCALE:(x + LR_PATCH) * SCALE])
        if random.random() < .5:
            a, b = a[:, ::-1], b[:, ::-1]
        if random.random() < .5:
            a, b = a[::-1], b[::-1]
        if random.random() < .5:
            a, b = a.transpose(1, 0, 2), b.transpose(1, 0, 2)
        return to_tensor(a), to_tensor(b)

## 5. Metrik
Foto yang lebih kecil dari 4× gambar kiriman (misalnya Kodak 768×512) dinilai pada ukuran aslinya: hasil semua metode diperkecil ke ukuran foto asli dulu.

In [9]:
try:
    import lpips
    LPIPS_NET = lpips.LPIPS(net="alex", verbose=False).to(DEVICE).eval()
except Exception as e:
    LPIPS_NET = None
    print("LPIPS dilewati:", e)
try:
    from skimage.metrics import structural_similarity
except Exception as e:
    structural_similarity = None
    print("SSIM dilewati:", e)


def luma(a):
    return 16 + (65.481 * a[..., 0] + 128.553 * a[..., 1] + 24.966 * a[..., 2]) / 255.0


def psnr(a, b):
    return 10 * math.log10(255 ** 2 / max(np.mean((luma(a.astype(float)) - luma(b.astype(float))) ** 2), 1e-10))


def ssim(a, b):
    return structural_similarity(luma(a.astype(float)), luma(b.astype(float)), data_range=255) if structural_similarity else float("nan")


@torch.no_grad()
def lpips_of(a, b):
    if LPIPS_NET is None:
        return float("nan")
    t = lambda x: (to_tensor(x)[None].to(DEVICE) * 2 - 1)
    return float(LPIPS_NET(t(a), t(b)))


def to_uint8(t):
    return (t.clamp(0, 1)[0].permute(1, 2, 0).cpu().numpy() * 255).round().astype(np.uint8)


def fit(out, gt):
    if out.shape != gt.shape:
        out = np.asarray(Image.fromarray(out).resize((gt.shape[1], gt.shape[0]), Image.LANCZOS))
    return out


def bicubic(lr):
    return np.asarray(Image.fromarray(lr).resize((lr.shape[1] * SCALE, lr.shape[0] * SCALE), Image.BICUBIC))


def model_fn(model):
    @torch.no_grad()
    def run(lr):
        model.eval()
        return to_uint8(model(to_tensor(lr)[None].to(DEVICE)))
    return run


def evaluate(fn, pairs, full=True, key=preset_of):
    rows = {}
    for p in pairs:
        lr, gt = np.load(p), np.load(gt_path(p))
        out = fit(fn(lr), gt)
        row = [psnr(gt, out)] + ([ssim(gt, out), lpips_of(gt, out)] if full else [])
        rows.setdefault(key(p), []).append(row)
    return {k: np.mean(v, axis=0) for k, v in sorted(rows.items())}


val_sample = random.Random(SEED).sample(val_pairs, min(VAL_PAIRS, len(val_pairs)))

/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 159MB/s]


## 6. Training
Checkpoint `NusaSR2/last.pt` ditulis setiap validasi. Run all lagi akan melanjutkan dari situ. Validasi memakai DIV2K valid (asli + kondisi lapangan), bukan foto uji.

In [10]:
model = (load_v1() if HAS_V1 else load_pretrained()).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR_FROM_V1 if HAS_V1 else LR_FROM_BASE, betas=(.9, .99))
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ITERS, eta_min=1e-6)
start, best = 0, -1.0
last_ckpt, best_ckpt = SAVE / "last.pt", SAVE / "best.pt"

if last_ckpt.exists():
    state = torch.load(last_ckpt, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state["model"]); opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"])
    start, best = state["iter"], state["best"]
    print(f"lanjut dari iterasi {start}, PSNR terbaik {best:.3f}")


def overall(result):
    return float(np.mean([v[0] for v in result.values()]))


if start == 0:
    best = overall(evaluate(model_fn(model), val_sample, full=False))
    print(f"awal ({'v1' if HAS_V1 else 'bobot bawaan'}): PSNR {best:.3f}")
    torch.save(model.state_dict(), best_ckpt)

groups = [(train_pairs, 1 - OWN_SHARE if own_pairs else 1.0), (own_pairs, OWN_SHARE)]
loader = DataLoader(Patches(groups, max(ITERS - start, 0) * BATCH), batch_size=BATCH, num_workers=NUM_WORKERS,
                    pin_memory=DEVICE == "cuda", persistent_workers=NUM_WORKERS > 0, drop_last=True)
model.train()
it, seen, loss_sum, t0 = start, 0, 0.0, time.time()
for lr_batch, gt_batch in loader:
    lr_batch, gt_batch = lr_batch.to(DEVICE, non_blocking=True), gt_batch.to(DEVICE, non_blocking=True)
    loss = F.l1_loss(model(lr_batch), gt_batch)
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    it += 1; seen += 1; loss_sum += float(loss)
    if it % 200 == 0 or it == ITERS:
        eta = (time.time() - t0) / seen * (ITERS - it) / 60
        print(f"iter {it}/{ITERS}  L1 {loss_sum / min(seen, 200):.5f}  sisa ±{eta:.0f} menit", flush=True)
        loss_sum = 0.0
    if it % VAL_EVERY == 0 or it == ITERS:
        score = overall(evaluate(model_fn(model), val_sample, full=False))
        if score > best:
            best = score
            torch.save(model.state_dict(), best_ckpt)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "iter": it, "best": best}, last_ckpt)
        print(f"validasi {it}: PSNR {score:.3f} (terbaik {best:.3f})", flush=True)
        model.train()
print("selesai, PSNR validasi terbaik:", round(best, 3))

awal (v1): PSNR 26.374


/tmp/ipykernel_5297/300002550.py:32: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  it += 1; seen += 1; loss_sum += float(loss)


iter 200/30000  L1 0.04446  sisa ±66 menit
iter 400/30000  L1 0.04534  sisa ±64 menit
iter 600/30000  L1 0.04519  sisa ±63 menit
iter 800/30000  L1 0.04516  sisa ±63 menit
iter 1000/30000  L1 0.04428  sisa ±63 menit
iter 1200/30000  L1 0.04470  sisa ±63 menit
iter 1400/30000  L1 0.04547  sisa ±63 menit
iter 1600/30000  L1 0.04488  sisa ±63 menit
iter 1800/30000  L1 0.04502  sisa ±62 menit
iter 2000/30000  L1 0.04603  sisa ±62 menit
validasi 2000: PSNR 26.377 (terbaik 26.377)
iter 2200/30000  L1 0.04549  sisa ±63 menit
iter 2400/30000  L1 0.04552  sisa ±63 menit
iter 2600/30000  L1 0.04553  sisa ±62 menit
iter 2800/30000  L1 0.04520  sisa ±62 menit
iter 3000/30000  L1 0.04588  sisa ±61 menit
iter 3200/30000  L1 0.04598  sisa ±61 menit
iter 3400/30000  L1 0.04578  sisa ±60 menit
iter 3600/30000  L1 0.04552  sisa ±60 menit
iter 3800/30000  L1 0.04549  sisa ±60 menit
iter 4000/30000  L1 0.04526  sisa ±59 menit
validasi 4000: PSNR 26.370 (terbaik 26.377)
iter 4200/30000  L1 0.04501  sisa ±5

## 7. Perbandingan
PSNR/SSIM makin tinggi makin baik, LPIPS makin rendah makin baik. Model v2 hanya layak dipakai kalau menang di foto lapangan dan tidak kalah jauh di kondisi `asli`.

In [11]:
tuned = SRVGGNetCompact().to(DEVICE)
tuned.load_state_dict(torch.load(best_ckpt, map_location=DEVICE, weights_only=True))
base = load_pretrained().to(DEVICE)
methods = [("bicubic", bicubic), ("x4v3 bawaan", model_fn(base))]
if HAS_V1:
    methods.append(("v1 (DIV2K)", model_fn(load_v1().to(DEVICE))))
methods.append(("v2 (lapangan)", model_fn(tuned)))
lines = []


def table(title, pairs, methods, key, key_name):
    start_line = len(lines)
    lines.append(f"\n== {title} ({len(pairs)} pasangan) ==")
    lines.append(f"{'metode':<24}{key_name:>8}{'PSNR':>9}{'SSIM':>9}{'LPIPS':>9}")
    for name, fn in methods:
        for k, (p, s, l) in evaluate(fn, pairs, key=key).items():
            lines.append(f"{name:<24}{k:>8}{p:>9.3f}{s:>9.4f}{l:>9.4f}")
    print("\n".join(lines[start_line:]), flush=True)


table("Kodak per kondisi (rata-rata 3 preset)", kodak_pairs, methods, kind_of, "kondisi")
table("Kodak per preset (semua kondisi)", kodak_pairs, methods, preset_of, "preset")
table("DIV2K valid per preset", val_pairs, methods, preset_of, "preset")
if field_pairs:
    table("Foto lapangan per preset", field_pairs, methods, preset_of, "preset")
else:
    lines.append("\nFoto lapangan belum ada: hasil belum mewakili kondisi nyata.")


== Kodak per kondisi (rata-rata 3 preset) (432 pasangan) ==
metode                   kondisi     PSNR     SSIM    LPIPS
bicubic                     asli   26.922   0.7037   0.5083
bicubic                    gelap   33.650   0.8110   0.4863
bicubic                   goyang   31.419   0.8229   0.3551
bicubic                    kabut   31.100   0.7905   0.4302
bicubic                    silau   25.906   0.6849   0.4917
bicubic                    warna   26.934   0.7041   0.4900
x4v3 bawaan                 asli   26.878   0.7184   0.3638
x4v3 bawaan                gelap   33.371   0.8159   0.3976
x4v3 bawaan               goyang   29.636   0.8158   0.2848
x4v3 bawaan                kabut   30.489   0.7946   0.3251
x4v3 bawaan                silau   25.881   0.7004   0.3567
x4v3 bawaan                warna   26.884   0.7188   0.3572
v1 (DIV2K)                  asli   27.959   0.7376   0.4008
v1 (DIV2K)                 gelap   34.321   0.8280   0.4227
v1 (DIV2K)                goyang   32.5

In [12]:
def sample_sheet(path, preset=256, kinds=("asli", "gelap", "kabut", "goyang")):
    """Gambar contoh: satu baris per foto+kondisi, kolom = metode lalu foto asli."""
    rows = [p for p in kodak_pairs if preset_of(p) == preset and kind_of(p) in kinds
            and any(Path(p).name.startswith(s + "-") for s in SAMPLES)]
    if not rows:
        return
    cell_w = 360
    names = [n for n, _ in methods] + ["foto asli"]
    first = np.load(gt_path(rows[0]))
    cell_h = int(cell_w * first.shape[0] / first.shape[1])
    sheet = Image.new("RGB", (cell_w * len(names), (cell_h + 20) * len(rows) + 20), "white")
    draw = ImageDraw.Draw(sheet)
    for j, n in enumerate(names):
        draw.text((j * cell_w + 4, 4), n, fill="black")
    for i, p in enumerate(rows):
        lr, gt = np.load(p), np.load(gt_path(p))
        outs = [fit(fn(lr), gt) for _, fn in methods] + [gt]
        y = 20 + i * (cell_h + 20)
        draw.text((4, y + 2), f"{Path(p).stem.rsplit('_', 2)[0]}  {preset}px", fill="black")
        for j, o in enumerate(outs):
            im = Image.fromarray(o)
            box = (im.width // 4, im.height // 4, im.width * 3 // 4, im.height * 3 // 4)  # potongan tengah, diperbesar
            sheet.paste(im.crop(box).resize((cell_w, cell_h), Image.LANCZOS), (j * cell_w, y + 18))
    sheet.save(path, quality=90)
    print("contoh:", path)


sample_sheet(SAVE / "contoh.jpg")

contoh: /content/drive/MyDrive/NusaSR2/contoh.jpg


## 8. Ekspor ke TFLite untuk app
Format sama dengan model bawaan: input `uint8` 128×128 RGB, output `uint8` 512×512 RGB. Tabel terakhir membandingkan v2 float dengan v2 8-bit: selisihnya adalah kualitas yang hilang karena kuantisasi.

In [13]:
import tensorflow as tf
keras = tf.keras


class DepthToSpace(keras.layers.Layer):
    def call(self, x):
        return tf.nn.depth_to_space(x, SCALE)


def build_keras(size=128, feats=64, convs=32):
    inp = keras.Input((size, size, 3), batch_size=1)
    x, conv_layers, prelu_layers = inp, [], []
    for i in range(convs + 1):
        conv, prelu = keras.layers.Conv2D(feats, 3, padding="same"), keras.layers.PReLU(shared_axes=[1, 2])
        x = prelu(conv(x)); conv_layers.append(conv); prelu_layers.append(prelu)
    last = keras.layers.Conv2D(3 * SCALE * SCALE, 3, padding="same")
    x = DepthToSpace()(last(x))
    out = keras.layers.Add()([x, keras.layers.UpSampling2D(SCALE, interpolation="nearest")(inp)])
    return keras.Model(inp, out), conv_layers + [last], prelu_layers


def port_weights(state, conv_layers, prelu_layers):
    n = lambda k: state[k].detach().cpu().numpy()
    for i, conv in enumerate(conv_layers):
        idx = 2 * i
        kernel, bias = n(f"body.{idx}.weight").transpose(2, 3, 1, 0), n(f"body.{idx}.bias")
        if conv is conv_layers[-1]:  # PyTorch PixelShuffle (C,i,j) ke TF depth_to_space (i,j,C)
            order = [c * SCALE ** 2 + i2 * SCALE + j for i2 in range(SCALE) for j in range(SCALE) for c in range(3)]
            kernel, bias = kernel[..., order], bias[order]
        conv.set_weights([kernel, bias])
    for i, prelu in enumerate(prelu_layers):
        prelu.set_weights([n(f"body.{2 * i + 1}.weight").reshape(1, 1, -1)])


def pad_tile(a):
    return np.pad(a, ((0, 128 - a.shape[0]), (0, 128 - a.shape[1]), (0, 0)), mode="edge")


def calibration_patches():
    rng = random.Random(SEED)
    pool = val_pairs + own_pairs  # termasuk kondisi gelap/kabut, supaya rentang nilainya terwakili
    for _ in range(CALIBRATION):
        lr = np.load(rng.choice(pool), mmap_mode="r")
        h, w = lr.shape[:2]
        y, x = rng.randint(0, max(h - 128, 0)), rng.randint(0, max(w - 128, 0))
        yield pad_tile(np.array(lr[y:y + 128, x:x + 128]))


def export_tflite(net):
    km, convs, prelus = build_keras()
    port_weights(net.state_dict(), convs, prelus)
    probe = np.random.default_rng(SEED).random((1, 128, 128, 3), dtype=np.float32)
    net.eval()
    with torch.no_grad():
        want = net(torch.from_numpy(probe).permute(0, 3, 1, 2).to(DEVICE)).permute(0, 2, 3, 1).cpu().numpy()
    diff = float(np.abs(km(probe, training=False).numpy() - want).max())
    assert diff < 1e-3, f"bobot Keras tidak cocok: selisih {diff}"
    conv = tf.lite.TFLiteConverter.from_keras_model(km)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = lambda: ([p[None].astype(np.float32) / 255.0] for p in calibration_patches())
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.uint8
    conv.inference_output_type = tf.uint8
    return conv.convert()


def tflite_fn(blob):
    it = tf.lite.Interpreter(model_content=blob, num_threads=THREADS)
    it.allocate_tensors()
    inp, out = it.get_input_details()[0], it.get_output_details()[0]
    in_scale, in_zero = inp["quantization"]
    out_scale, out_zero = out["quantization"]

    def tile(a):
        h, w = a.shape[:2]
        q = np.clip(np.round(pad_tile(a).astype(np.float32) / 255.0 / in_scale + in_zero), 0, 255).astype(np.uint8)
        it.set_tensor(inp["index"], q[None]); it.invoke()
        o = (it.get_tensor(out["index"])[0].astype(np.float32) - out_zero) * out_scale * 255.0
        return np.clip(o, 0, 255).round().astype(np.uint8)[:h * SCALE, :w * SCALE]

    def run(a):
        h, w = a.shape[:2]
        if max(h, w) <= 128:
            return tile(a)
        out_img, step = np.zeros((h * SCALE, w * SCALE, 3), np.uint8), 128 - 2 * OVERLAP
        for y in range(0, h, step):
            for x in range(0, w, step):
                sy, sx = max(0, y - OVERLAP), max(0, x - OVERLAP)
                t = tile(a[sy:min(h, y + step + OVERLAP), sx:min(w, x + step + OVERLAP)])
                ch, cw = min(step, h - y), min(step, w - x)
                out_img[y * SCALE:(y + ch) * SCALE, x * SCALE:(x + cw) * SCALE] = \
                    t[(y - sy) * SCALE:(y - sy + ch) * SCALE, (x - sx) * SCALE:(x - sx + cw) * SCALE]
        return out_img
    return run, inp, out


base_blob, tuned_blob = export_tflite(base), export_tflite(tuned)
tuned_path = SAVE / "real_esrgan_x4v3_lapangan_w8a8.tflite"
tuned_path.write_bytes(tuned_blob)

base_run, _, _ = tflite_fn(base_blob)
tuned_run, inp, out = tflite_fn(tuned_blob)
assert list(inp["shape"]) == [1, 128, 128, 3] and inp["dtype"] == np.uint8
assert list(out["shape"]) == [1, 512, 512, 3] and out["dtype"] == np.uint8

pool = field_pairs or kodak_pairs
test_pairs = random.Random(SEED).sample(pool, min(TFLITE_IMAGES * len(PRESETS), len(pool)))
table("TFLite 8-bit dengan tiling seperti app", test_pairs,
      [("x4v3 bawaan 8-bit", base_run), ("v2 float", model_fn(tuned)), ("v2 8-bit", tuned_run)], preset_of, "preset")
(SAVE / "hasil.txt").write_text("\n".join(lines), encoding="utf-8")
print("\nTersimpan:", tuned_path, f"({tuned_path.stat().st_size / 1024:.0f} KB)")
print("Kirim isi", SAVE / "hasil.txt", "dan", SAVE / "contoh.jpg", "untuk dianalisis.")

Saved artifact at '/tmp/tmpt7mhve_w'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 128, 128, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(1, 512, 512, 3), dtype=tf.float32, name=None)
Captures:
  133498689020624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689021584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689020432: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689021200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689022352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689021392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689022160: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689022736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689019088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133498689020816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1334986890

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Saved artifact at '/tmp/tmprov9vz6r'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(1, 128, 128, 3), dtype=tf.float32, name='keras_tensor_71')
Output Type:
  TensorSpec(shape=(1, 512, 512, 3), dtype=tf.float32, name=None)
Captures:
  133497776095312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774117264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774116112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774116880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774117840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774116304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774117456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774118416: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774117072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133497774118032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1334977

/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



== TFLite 8-bit dengan tiling seperti app (30 pasangan) ==
metode                    preset     PSNR     SSIM    LPIPS
x4v3 bawaan 8-bit            160   27.158   0.6602   0.3869
x4v3 bawaan 8-bit            256   28.213   0.6879   0.3249
x4v3 bawaan 8-bit            384   29.561   0.8025   0.2418
v2 float                     160   29.119   0.7169   0.4689
v2 float                     256   29.732   0.7374   0.4217
v2 float                     384   32.453   0.8557   0.2467
v2 8-bit                     160   28.527   0.6905   0.4006
v2 8-bit                     256   29.208   0.7170   0.3405
v2 8-bit                     384   31.768   0.8415   0.2269

Tersimpan: /content/drive/MyDrive/NusaSR2/real_esrgan_x4v3_lapangan_w8a8.tflite (1284 KB)
Kirim isi /content/drive/MyDrive/NusaSR2/hasil.txt dan /content/drive/MyDrive/NusaSR2/contoh.jpg untuk dianalisis.


## 9. Kalau hasilnya menang

Unduh `real_esrgan_x4v3_lapangan_w8a8.tflite`, `hasil.txt`, dan `contoh.jpg` dari Drive folder `NusaSR2`. Bentuk input/output sama dengan model bawaan, jadi model bisa langsung diuji di app tanpa ubah kode tiling.

Jangan pakai v2 kalau hanya menang di kondisi buatan tetapi kalah di foto lapangan.